# Model Development & Tracking 
Covers **Stage 2.2–2.4 + Stage 3.** Complete every `# TODO` in this notebook **and** in the `src/` modules it imports. 

- Each stage opens with a **sub-task checklist**
- Capture any repo-generated evidence (MLflow UI, Docker build, CI run, drift report) as screenshots/summaries **inside the notebook/report**.

**File ownership** — 
- *Provided:* `config.py`, `src/evaluate.py`. 
- *Provided to extend:* `src/model.py`, `src/train.py`, `app.py`. 
- *You build:* this notebook.

### 0. Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import config
from src import data_prep
from src.model import build_model, trainable_parameters

print(f"PyTorch: {torch.__version__}")
print(f"Device: {config.DEVICE}")

## **Stage 2.2 — Transfer-Learning Model** <font color="red">[7 marks]</font>

- **2.2.1 — ImageNet-pretrained ResNet18, backbone frozen [4]**
- **2.2.2 — New 2-class head replaces fc (head trained) [3]**

**Objective:** Build an ImageNet-pretrained ResNet18 with a frozen backbone + new 2-class head.

**Implement in:** src/model.build_model

**Inputs → Outputs:** → a torch model; only the head's params have requires_grad=True

**TODO:** load weights=IMAGENET1K + freeze the backbone (2.2.1); replace fc with a 2-class Linear head so only the head trains (2.2.2); print total vs trainable params.

**Depends on:** Stage 2.1 transformations completed in Data_Preparation.ipynb  ·  **Document here:** the param counts (trainable should be ~the head only).

In [ ]:
# 2.2.1-2.2.2: Build model + print params
net = build_model(freeze=config.FREEZE_BACKBONE)

total_params = sum(p.numel() for p in net.parameters())
trainable = trainable_parameters(net)
train_params = sum(p.numel() for p in trainable)

print(f"Model: {config.BACKBONE}")
print(f"Total parameters:     {total_params:>10,}")
print(f"Trainable parameters: {train_params:>10,}")
print(f"Frozen parameters:    {total_params - train_params:>10,}")
print(f"\nFinal layer (head):")
print(f"  {net.fc}")
print(f"\nBackbone frozen: {config.FREEZE_BACKBONE}")
print(f"Only the head trains: {all(not p.requires_grad for n, p in net.named_parameters() if 'fc' not in n)}")

## **Stage 2.3 — Training Workflow** <font color="red">[6 marks]</font>

- **2.3.1 — Class-weighted loss + optimiser (trainable params) + seed [3]**
- **2.3.2 — Epoch loop + validation eval + early stopping [3]**

**Objective:** Train the head reproducibly with validation + early stopping.

**Implement in:** src/train.py

**Inputs → Outputs:** splits + model → trained model.pt + model_meta.json

**TODO:** class-weighted CrossEntropy + Adam over TRAINABLE params only + seed 42 (2.3.1); epoch loop with val evaluation + early stop on val-F1 (2.3.2).

**Document here:** the training summary (final val F1, epochs).

In [ ]:
# 2.3.1-2.3.2: Run training pipeline
!python -m src.train

In [ ]:
# Load and display training metadata
model_meta = json.loads(config.MODEL_META_PATH.read_text())
print("=== Training Summary ===")
print(json.dumps(model_meta, indent=2))

## **Stage 2.4 — MLflow Experiment Tracking** <font color="red">[6 marks]</font>

- **2.4.1 — Params + per-epoch metrics logged [3]**
- **2.4.2 — Trained model logged to the run [3]**

**Objective:** Track the experiment in MLflow.

**Implement in:** src/train.py (mlflow calls)

**Inputs → Outputs:** training run → MLflow params + per-epoch metrics + logged model

**TODO:** set_experiment; log_params; log_metrics(step=epoch) (2.4.1); log_model (2.4.2).

**Document here:** the runs table (mlflow.search_runs).

In [ ]:
# 2.4.1-2.4.2: Display MLflow runs
import mlflow
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)

runs = mlflow.search_runs(experiment_names=[config.MLFLOW_EXPERIMENT])
print(f"Total runs: {len(runs)}")
display_cols = [c for c in runs.columns if c.startswith('params.') or c.startswith('metrics.test_') or c == 'run_id' or c == 'status']
print(runs[display_cols].to_string())

## **Stage 3.1 — Evaluation & Failure Analysis** <font color="red">[7 marks]</font>

- **3.1.1 — Imbalance-aware metrics + model selection [4]**
- **3.1.2 — Confusion/ROC plots + failure-case analysis [3]**

**Objective:** Evaluate on defect recall/F1, not accuracy.

**Implement in:** src/evaluate.py

**Inputs → Outputs:** model + test split → metrics.json + model_eval.png

**TODO:** report recall(defect)/precision/F1/ROC-AUC + select on recall/F1 (3.1.1); plot ROC + confusion and inspect misclassified samples (3.1.2).

**Document here:** the metrics + a short interpretation (why recall matters here).

In [ ]:
# 3.1.1: Load and display test metrics
metrics = json.loads(config.METRICS_PATH.read_text())
print("=== Test Metrics ===")
for k, v in metrics.items():
    if k != 'confusion_matrix':
        print(f"  {k}: {v:.4f}")
print(f"\nConfusion Matrix:")
print(f"  {metrics['confusion_matrix']}")

print("\n=== Why Recall Matters ===")
print("In casting defect detection, missing a defective part (false negative)")
print("is far more costly than flagging a good part (false positive).")
print("Recall on defects is the headline metric for quality control.")

In [ ]:
# 3.1.2: Show evaluation plots
from IPython.display import display, Image as IPImage
eval_plot = config.ARTIFACT_DIR / 'model_eval.png'
if eval_plot.exists():
    display(IPImage(filename=str(eval_plot)))
else:
    print("model_eval.png not found — run training first.")

## **Stage 3.2 — Model Registry & Promotion** <font color="red">[5 marks]</font>

- **3.2.1 — Best model registered in MLflow Registry [2]**
- **3.2.2 — Promotion to production alias + version history [3]**

**Objective:** Register and promote the best model.

**Implement in:** src/train.py (MLflow registry)

**Inputs → Outputs:** best run → registered model + @production alias

**TODO:** register_model(casting_defect_classifier) (3.2.1); set_registered_model_alias('production', v) with version history (3.2.2).

**Document here:** the registry version + alias (screenshot the MLflow Models page).

In [ ]:
# 3.2.1-3.2.2: Show registered model + production alias
from mlflow import MlflowClient
client = MlflowClient()

try:
    mv = client.get_model_version_by_alias(config.REGISTERED_MODEL, config.PRODUCTION_ALIAS)
    print(f"Registered Model: {config.REGISTERED_MODEL}")
    print(f"Production Alias: @{config.PRODUCTION_ALIAS}")
    print(f"Version: {mv.version}")
    print(f"Run ID: {mv.run_id}")
    print(f"Status: {mv.status}")
    
    # Version history
    print(f"\n=== Version History ===")
    versions = client.search_model_versions(f"name='{config.REGISTERED_MODEL}'")
    for v in versions:
        aliases = v.aliases if hasattr(v, 'aliases') else []
        print(f"  v{v.version} — run={v.run_id[:8]}... aliases={aliases}")
except Exception as e:
    print(f"Registry not available yet: {e}")
    print("Run `python -m src.train` first.")

## **Stage 3.3 — FastAPI Inference Service** <font color="red">[6 marks]</font>

- **3.3.1 — Image /predict + /health + logging + validation [4]** — *to be done in app.py*
- **3.3.2 — Live call demonstrated in-notebook [2]** — *to be done in this notebook*

**Objective:** Serve the model behind an image API and demonstrate it.

**Implement in:** app.py (+ a TestClient demo here)

**Inputs → Outputs:** image upload → {label, prob_defect, confidence}; /health; predictions logged

**TODO:** implement /health + POST /predict (UploadFile) with validation (bad image 400, no-model 503) + logging (3.3.1, in app.py); demo with TestClient on a sample image (3.3.2, here).

**Document here:** the live request/response (TestClient output).

In [ ]:
# 3.3.2: TestClient demo
from fastapi.testclient import TestClient
import app

with TestClient(app.app) as client:
    # Test /health
    resp = client.get('/health')
    print('=== GET /health ===')
    print(json.dumps(resp.json(), indent=2))
    
    # Test /predict with a sample image
    root = data_prep.find_data_root()
    sample = sorted((root / 'test' / 'def_front').glob('*.jpeg'))[0]
    print(f'\n=== POST /predict ({sample.name}) ===')
    with open(sample, 'rb') as f:
        resp = client.post('/predict', files={'file': (sample.name, f, 'image/jpeg')})
    print(json.dumps(resp.json(), indent=2))
    
    # Test with an OK image
    sample_ok = sorted((root / 'test' / 'ok_front').glob('*.jpeg'))[0]
    print(f'\n=== POST /predict ({sample_ok.name}) ===')
    with open(sample_ok, 'rb') as f:
        resp = client.post('/predict', files={'file': (sample_ok.name, f, 'image/jpeg')})
    print(json.dumps(resp.json(), indent=2))

## **Stage 3.4 — Containerisation & CI/CD** <font color="red">[7 marks]</font>

- **3.4.1 — Valid Dockerfile [3]** — *to be done in this notebook and the report*
- **3.4.2 — GitHub Actions CI + passing-test evidence [4]** — *to be done in this notebook and the report*

**Objective:** Containerise and automate tests.

**Implement in:** Dockerfile + .github/workflows/ci.yml (provided — review + evidence them)

**Inputs → Outputs:** code → Docker image; push/PR → CI runs pytest + docker build

**TODO:** show the Dockerfile (3.4.1); run pytest here + include a screenshot of the green CI run and successful docker build (3.4.2).

**Document here:** pytest output + Docker/CI screenshots **(Note: Don't add a repo link)**

In [ ]:
# 3.4.1: Display Dockerfile
print('=== Dockerfile ===')
print(Path('Dockerfile').read_text())

# Display CI workflow
ci_path = Path('.github/workflows/ci.yml')
if ci_path.exists():
    print('\n=== .github/workflows/ci.yml ===')
    print(ci_path.read_text())

In [ ]:
# 3.4.2: Run pytest
!pytest -q